In [5]:
# 1. INITIAL SETUP
!pip install yfinance pandas matplotlib plotly fredapi ipywidgets
%matplotlib inline

import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from fredapi import Fred
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display

# Initialize FRED API
fred = Fred(api_key='ffa1c3fabd429889449e01d7e3cd1c5f')  # Replace with your FRED API key

Defaulting to user installation because normal site-packages is not writeable


In [6]:
# 2. DATE RANGE SELECTION
start_date = widgets.DatePicker(
    description='Start Date',
    value=datetime.now() - timedelta(days=365*5)  # Default 5 years back
)
end_date = widgets.DatePicker(
    description='End Date',
    value=datetime.now()  # Default today
)

display(widgets.HBox([start_date, end_date]))

In [8]:
# 3. DATA ACQUISITION (RUN AFTER SETTING DATES)
def get_all_data(start, end):
    # Convert dates to strings
    start_str = start.strftime('%Y-%m-%d')
    end_str = end.strftime('%Y-%m-%d')
    
    # Fetch and normalize inflation data
    pce = fred.get_series('PCE', start_str, end_str)
    core_pce = fred.get_series('PCEPILFE', start_str, end_str)
    cpi = fred.get_series('CPIAUCSL', start_str, end_str)
    
    inflation = pd.concat([pce, core_pce, cpi], axis=1)
    inflation.columns = ['PCE', 'Core PCE', 'CPI']
    inflation = (inflation / inflation.iloc[0] * 100).round(2)  # Normalize to 100
    
    # Fetch yield data
    two_yr = fred.get_series('DGS2', start_str, end_str)
    ten_yr = fred.get_series('DGS10', start_str, end_str)
    thirty_yr = fred.get_series('DGS30', start_str, end_str)
    
    yields = pd.concat([two_yr, ten_yr, thirty_yr], axis=1)
    yields.columns = ['2Y', '10Y', '30Y']
    yields['10Y-2Y Spread'] = yields['10Y'] - yields['2Y']
    
    # Fetch liquidity data
    fed_bs = fred.get_series('WALCL', start_str, end_str)
    m2 = fred.get_series('M2SL', start_str, end_str)
    
    liquidity = pd.concat([fed_bs, m2], axis=1)
    liquidity.columns = ['Fed Balance Sheet', 'M2 Money Supply']
    
    # Fetch S&P 500 - UPDATED TO HANDLE NEW YFINANCE FORMAT
    try:
        sp500_data = yf.download('^GSPC', start=start_str, end=end_str)
        # Try common column names
        sp500 = sp500_data.get('Adj Close', sp500_data.get('Close'))
        if sp500 is None:
            raise KeyError("Could not find price column")
    except Exception as e:
        print(f"Error fetching S&P 500 data: {e}")
        sp500 = pd.Series(dtype=float)  # Return empty series if fails
    
    return {
        'inflation': inflation,
        'yields': yields,
        'liquidity': liquidity,
        'sp500': sp500
    }

# Execute after setting dates
data = get_all_data(start_date.value, end_date.value)

[*********************100%***********************]  1 of 1 completed


In [10]:
# 4. VISUALIZATION DASHBOARD
def create_dashboard(data):
    fig = make_subplots(
        rows=5, cols=1,
        shared_xaxes=True,
        subplot_titles=(
            "Normalized Inflation Indicators (Base=100)",
            "Treasury Yields", 
            "Yield Spread (10Y-2Y)",
            "Liquidity Indicators",
            "S&P 500"
        ),
        vertical_spacing=0.08
    )
    
    # Inflation (normalized)
    for col in data['inflation'].columns:
        fig.add_trace(
            go.Scatter(
                x=data['inflation'].index,
                y=data['inflation'][col],
                name=col,
                mode='lines'
            ),
            row=1, col=1
        )
    
    # Yields
    for col in ['2Y', '10Y', '30Y']:
        fig.add_trace(
            go.Scatter(
                x=data['yields'].index,
                y=data['yields'][col],
                name=col,
                mode='lines'
            ),
            row=2, col=1
        )
    
    # Spread
    fig.add_trace(
        go.Scatter(
            x=data['yields'].index,
            y=data['yields']['10Y-2Y Spread'],
            name='10Y-2Y Spread',
            line=dict(color='red')
        ),
        row=3, col=1
    )
    fig.add_hline(y=0, line_dash="dot", row=3, col=1)
    
    # Liquidity
    fig.add_trace(
        go.Scatter(
            x=data['liquidity'].index,
            y=data['liquidity']['Fed Balance Sheet'],
            name='Fed Balance Sheet',
            line=dict(color='blue')
        ),
        row=4, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=data['liquidity'].index,
            y=data['liquidity']['M2 Money Supply'],
            name='M2 Money Supply',
            line=dict(color='green')
        ),
        row=4, col=1
    )
    
    # S&P 500 (with existence check)
    if not data['sp500'].empty:
        fig.add_trace(
            go.Scatter(
                x=data['sp500'].index,
                y=data['sp500'],
                name='S&P 500',
                line=dict(color='purple')
            ),
            row=5, col=1
        )
    else:
        fig.add_annotation(
            row=5, col=1,
            text="S&P 500 data not available",
            showarrow=False
        )
    
    # Update layout (FIXED DATE FORMATTING)
    fig.update_layout(
        height=1400,
        width=1000,
        title_text=f"Market Dashboard ({start_date.value} to {end_date.value})",  # Removed .date()
        hovermode="x unified",
        showlegend=True
    )
    
    return fig

# Display dashboard
dashboard = create_dashboard(data)
dashboard.show()